In [1]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import date

In [2]:

import numpy as np
from datetime import timedelta

In [3]:

load_dotenv()
NASA_API_KEY = os.getenv("NASA_API_KEY")

In [4]:

# Fetch data
def fetch_solar_flares(start_date, end_date):
    url = "https://api.nasa.gov/DONKI/FLR"
    params = {"startDate": start_date, "endDate": end_date, "api_key": NASA_API_KEY}
    response = requests.get(url, params=params)
    return response.json()



In [5]:
flares = fetch_solar_flares("2020-01-01", str(date.today()))

In [6]:

print(f"Total flares fetched: {len(flares)}")

Total flares fetched: 2689


In [7]:
df = pd.DataFrame(flares)


In [8]:
df.sample(5)

,flrID,catalog,instruments,beginTime,peakTime,endTime,classType,sourceLocation,activeRegionNum,note,submissionTime,versionId,link,linkedEvents,sentNotifications
1734,2024-10-03T19:18:00-FLR-001,M2M_CATALOG,[{'displayName': 'GOES-P: EXIS 1.0-8.0'}],2024-10-03T19:18Z,2024-10-03T19:21Z,2024-10-03T19:24Z,M1.1,S15W27,13844.0,,2024-10-03T23:03Z,1,https://webtools.ccmc.gsfc.nasa.gov/DONKI/view...,None,None
674,2023-07-10T22:06:00-FLR-001,M2M_CATALOG,[{'displayName': 'GOES-P: EXIS 1.0-8.0'}],2023-07-10T22:06Z,2023-07-10T22:18Z,2023-07-10T22:29Z,M1.4,S13W62,13358.0,,2023-07-11T12:21Z,1,https://webtools.ccmc.gsfc.nasa.gov/DONKI/view...,None,None
2456,2025-12-09T01:34:00-FLR-001,M2M_CATALOG,[{'displayName': 'GOES-P: EXIS 1.0-8.0'}],2025-12-09T01:34Z,2025-12-09T01:38Z,2025-12-09T01:42Z,M1.1,S19W41,14294.0,,2025-12-09T02:05Z,1,https://webtools.ccmc.gsfc.nasa.gov/DONKI/view...,[{'activityID': '2025-12-09T03:00:00-CME-001'}],None
215,2022-05-17T06:16:00-FLR-001,M2M_CATALOG,[{'displayName': 'GOES-P: EXIS 1.0-8.0'}],2022-05-17T06:16Z,2022-05-17T06:19Z,2022-05-17T06:24Z,C2.9,S16E21,13010.0,The signature of this short-duration flare is ...,2022-05-17T16:30Z,1,https://webtools.ccmc.gsfc.nasa.gov/DONKI/view...,[{'activityID': '2022-05-17T06:48:00-CME-001'}],None
270,2022-08-19T11:01:00-FLR-001,M2M_CATALOG,[{'displayName': 'GOES-P: EXIS 1.0-8.0'}],2022-08-19T11:01Z,2022-08-19T11:17Z,2022-08-19T11:32Z,C4.0,S24W53,13078.0,,2022-08-21T14:50Z,1,https://webtools.ccmc.gsfc.nasa.gov/DONKI/view...,[{'activityID': '2022-08-19T11:38:00-CME-001'}],None


In [9]:
df = df[['flrID', 'beginTime', 'peakTime', 'endTime', 'classType', 
         'sourceLocation', 'activeRegionNum', 'linkedEvents']]

In [10]:
df.sample(5)

,flrID,beginTime,peakTime,endTime,classType,sourceLocation,activeRegionNum,linkedEvents
288,2022-08-28T18:20:00-FLR-001,2022-08-28T18:20Z,2022-08-28T18:32Z,2022-08-28T18:50Z,M4.6,S27W87,13088.0,None
685,2023-07-12T08:49:00-FLR-001,2023-07-12T08:49Z,2023-07-12T08:55Z,2023-07-12T09:00Z,M6.9,N26E86,13372.0,None
2606,2026-02-25T06:41:00-FLR-001,2026-02-25T06:41Z,2026-02-25T06:56Z,2026-02-25T07:09Z,C2.6,S08W27,NaN,[{'activityID': '2026-02-25T07:38:00-CME-002'}]
1316,2024-06-01T08:26:00-FLR-001,2024-06-01T08:26Z,2024-06-01T08:48Z,2024-06-01T08:58Z,X1.4,S20E27,13697.0,None
239,2022-07-10T23:30:00-FLR-001,2022-07-10T23:30Z,2022-07-10T23:43Z,2022-07-10T23:52Z,M1.3,S19E71,13056.0,None


In [11]:

df['flare_class'] = df['classType'].str[0]          
df['flare_intensity'] = df['classType'].str[1:].astype(float)

In [12]:

df['beginTime'] = pd.to_datetime(df['beginTime'])
df['peakTime'] = pd.to_datetime(df['peakTime'])
df['endTime'] = pd.to_datetime(df['endTime'])

In [13]:

df['duration_minutes'] = (df['endTime'] - df['beginTime']).dt.total_seconds() / 60

In [14]:
df['has_linked_events'] = df['linkedEvents'].apply(lambda x: 1 if x else 0)

In [15]:
df = df.drop(columns=['flrID', 'linkedEvents', 'classType'])

In [16]:
print(df.isnull().sum())
print(df.shape)
print(df.head())

beginTime             0
peakTime              0
endTime              26
sourceLocation        0
activeRegionNum      87
flare_class           0
flare_intensity       0
duration_minutes     26
has_linked_events     0
dtype: int64
(2689, 9)
                  beginTime                  peakTime  \
0 2020-05-29 07:18:00+00:00 2020-05-29 07:54:00+00:00   
1 2020-11-20 16:44:00+00:00 2020-11-20 17:03:00+00:00   
2 2020-11-20 18:11:00+00:00 2020-11-20 18:41:00+00:00   
3 2020-11-29 12:34:00+00:00 2020-11-29 13:11:00+00:00   
4 2020-12-07 15:46:00+00:00 2020-12-07 16:32:00+00:00   

                    endTime sourceLocation  activeRegionNum flare_class  \
0                       NaT         N32E90              NaN           M   
1                       NaT         S22E40          12783.0           C   
2                       NaT        S25E120              NaN           C   
3 2020-11-29 13:41:00+00:00         S25E97              NaN           M   
4                       NaT         S23W11 

In [17]:
df['duration_minutes'] = df['duration_minutes'].fillna(df['duration_minutes'].median())


In [18]:
df['activeRegionNum'] = df['activeRegionNum'].fillna(0)

In [19]:

df = df.drop(columns=['endTime', 'peakTime'])

print(df.isnull().sum())
print(df.shape)

beginTime            0
sourceLocation       0
activeRegionNum      0
flare_class          0
flare_intensity      0
duration_minutes     0
has_linked_events    0
dtype: int64
(2689, 7)


In [20]:
df = df.sort_values('beginTime').reset_index(drop=True)

In [21]:

df['date'] = df['beginTime'].dt.date

In [22]:
df['is_significant'] = df['flare_class'].apply(lambda x: 1 if x in ['M', 'X'] else 0)

In [24]:

print(df['is_significant'].value_counts())

is_significant
1    2179
0     510
Name: count, dtype: int64


In [25]:
def check_next_24hrs(idx, df):
    current_time = df.loc[idx, 'beginTime']
    next_24 = current_time + timedelta(hours=24)
    
    # Check if any significant flare exists in next 24 hours
    future_flares = df[
        (df['beginTime'] > current_time) & 
        (df['beginTime'] <= next_24) &
        (df['is_significant'] == 1)
    ]
    return 1 if len(future_flares) > 0 else 0

In [26]:

df['target'] = [check_next_24hrs(i, df) for i in range(len(df))]

In [27]:

print(df['target'].value_counts())
print(f"\nHazardous %: {df['target'].mean() * 100:.1f}%")

target
1    2043
0     646
Name: count, dtype: int64

Hazardous %: 76.0%


In [28]:
from sklearn.preprocessing import LabelEncoder

# Encode flare_class (C, M, X → 0, 1, 2)
le = LabelEncoder()
df['flare_class_encoded'] = le.fit_transform(df['flare_class'])

# Extract time features from beginTime
df['hour'] = df['beginTime'].dt.hour
df['month'] = df['beginTime'].dt.month
df['dayofweek'] = df['beginTime'].dt.dayofweek

# Final feature set
features = [
    'flare_class_encoded',
    'flare_intensity',
    'duration_minutes',
    'activeRegionNum',
    'has_linked_events',
    'hour',
    'month',
    'dayofweek'
]

X = df[features]
y = df['target']

print(X.shape)
print(X.head())

(2689, 8)
   flare_class_encoded  flare_intensity  duration_minutes  activeRegionNum  \
0                    3              1.1              21.0              0.0   
1                    2              1.9              21.0          12783.0   
2                    2              1.0              21.0              0.0   
3                    3              4.4              67.0              0.0   
4                    2              7.4              21.0          12790.0   

   has_linked_events  hour  month  dayofweek  
0                  0     7      5          4  
1                  1    16     11          4  
2                  0    18     11          4  
3                  1    12     11          6  
4                  1    15     12          0  


In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
import mlflow
import mlflow.xgboost

mlflow.set_experiment("solar_flare_prediction")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Handle imbalance with scale_pos_weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

with mlflow.start_run():

    # Train XGBoost
    model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=scale,
        random_state=42,
        eval_metric='logloss'
    )

    model.fit(X_train, y_train)

    # Evaluate
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc_score = roc_auc_score(y_test, y_prob)

    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC Score: {auc_score:.4f}")

    # --- MLflow logging ---
    mlflow.log_params({
        "n_estimators": 100,
        "max_depth": 5,
        "learning_rate": 0.1,
        "scale_pos_weight": scale
    })
    mlflow.log_metric("roc_auc", auc_score)
    mlflow.xgboost.log_model(model, "model")

              precision    recall  f1-score   support

           0       0.50      0.66      0.57       129
           1       0.88      0.79      0.83       409

    accuracy                           0.76       538
   macro avg       0.69      0.72      0.70       538
weighted avg       0.79      0.76      0.77       538

ROC-AUC Score: 0.7961


In [30]:
print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"scale_pos_weight: {scale:.2f}")

Training set size: 2151
Test set size: 538
scale_pos_weight: 0.32


In [31]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Train XGBoost
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

              precision    recall  f1-score   support

           0       0.50      0.66      0.57       129
           1       0.88      0.79      0.83       409

    accuracy                           0.76       538
   macro avg       0.69      0.72      0.70       538
weighted avg       0.79      0.76      0.77       538

ROC-AUC Score: 0.7961


In [32]:
# from sklearn.model_calls import cross_val_score
from xgboost import XGBClassifier

# Try tuned model
model_v2 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='logloss'
)

model_v2.fit(X_train, y_train)

y_pred_v2 = model_v2.predict(X_test)
y_prob_v2 = model_v2.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_v2))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_v2):.4f}")

              precision    recall  f1-score   support

           0       0.50      0.64      0.56       129
           1       0.88      0.80      0.84       409

    accuracy                           0.76       538
   macro avg       0.69      0.72      0.70       538
weighted avg       0.79      0.76      0.77       538

ROC-AUC Score: 0.7955


In [33]:
# Add new features to df
df['intensity_x_duration'] = df['flare_intensity'] * df['duration_minutes']
df['is_extreme'] = df['flare_class'].apply(lambda x: 1 if x == 'X' else 0)
df['is_strong'] = df['flare_class'].apply(lambda x: 1 if x in ['M', 'X'] else 0)

# Updated feature set
features_v2 = [
    'flare_class_encoded',
    'flare_intensity',
    'duration_minutes',
    'activeRegionNum',
    'has_linked_events',
    'hour',
    'month',
    'dayofweek',
    'intensity_x_duration',  # new
    'is_extreme',             # new
    'is_strong'               # new
]

X_v2 = df[features_v2]
y_v2 = df['target']

# Retrain with original model params
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2, y_v2, test_size=0.2, random_state=42, stratify=y_v2
)

neg = (y_train_v2 == 0).sum()
pos = (y_train_v2 == 1).sum()
scale_v2 = neg / pos

model_v3 = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_v2,
    random_state=42,
    eval_metric='logloss'
)

model_v3.fit(X_train_v2, y_train_v2)

y_pred_v3 = model_v3.predict(X_test_v2)
y_prob_v3 = model_v3.predict_proba(X_test_v2)[:, 1]

print(classification_report(y_test_v2, y_pred_v3))
print(f"ROC-AUC Score: {roc_auc_score(y_test_v2, y_prob_v3):.4f}")

              precision    recall  f1-score   support

           0       0.48      0.65      0.55       129
           1       0.88      0.78      0.83       409

    accuracy                           0.75       538
   macro avg       0.68      0.72      0.69       538
weighted avg       0.78      0.75      0.76       538

ROC-AUC Score: 0.7894


In [34]:
import os
import joblib
import json

os.makedirs('../model', exist_ok=True)

joblib.dump(model, '../model/flare_model.joblib')
joblib.dump(le, '../model/label_encoder.joblib')

with open('../model/features.json', 'w') as f:
    json.dump(features, f)

print("Model saved successfully! ✅")

Model saved successfully! ✅


In [35]:
# Check what label encoder classes look like
print("Label encoder classes:", le.classes_)

# Check feature order
print("Features:", features)

# Check what encoded value X gets
print("X encoded:", le.transform(['X'])[0])
print("M encoded:", le.transform(['M'])[0])

# Check model feature importances
import pandas as pd
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
print(importance)

Label encoder classes: ['A' 'B' 'C' 'M' 'X']
Features: ['flare_class_encoded', 'flare_intensity', 'duration_minutes', 'activeRegionNum', 'has_linked_events', 'hour', 'month', 'dayofweek']
X encoded: 4
M encoded: 3
               feature  importance
0  flare_class_encoded    0.408411
4    has_linked_events    0.237544
3      activeRegionNum    0.093630
6                month    0.056878
2     duration_minutes    0.056518
7            dayofweek    0.053636
1      flare_intensity    0.048165
5                 hour    0.045219


In [36]:
# How many X flares actually have target=1?
print(df[df['flare_class'] == 'X']['target'].value_counts())

# How many M flares have target=1?
print(df[df['flare_class'] == 'M']['target'].value_counts())

target
1    95
0    10
Name: count, dtype: int64
target
1    1733
0     341
Name: count, dtype: int64
